# Week 60, Day 1: Closures

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w060d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def make_counter():
    count = 0
    def step():
        nonlocal count
        count += 1
        return count
    return step

a, b = make_counter(), make_counter()
print(a(), a(), a(), b())
print(a.__code__.co_freevars, a.__closure__[0].cell_contents)

In [ ]:
def make_pair():
    shared = []
    def add(x):
        shared.append(x)
    def show():
        return list(shared)
    return add, show

add, show = make_pair()
add("tea")
add("jam")
print(show(), add.__code__.co_freevars, add.__closure__[0] is show.__closure__[0])

def outer():
    x = 1
    def inner():
        return x
    x = 2                 # changed after inner was made: inner sees it
    return inner

print(outer()())

In [ ]:
def make_averager():
    total, count = 0, 0
    def add(value):
        nonlocal total, count
        total += value
        count += 1
        return total / count
    return add

avg = make_averager()
print(avg(10), avg(20), avg(60))

def make_log():
    lines = []
    def log(text):
        lines.append(text)            # mutating: no nonlocal needed
        return len(lines)
    return log

log = make_log()
print(log("start"), log("stop"))

In [ ]:
broken = [lambda x: x + i for i in range(3)]
with_default = [lambda x, i=i: x + i for i in range(3)]

def adder(i):
    return lambda x: x + i
with_factory = [adder(i) for i in range(3)]

print([f(10) for f in broken], [f(10) for f in with_default], [f(10) for f in with_factory])

In [ ]:
def memoised(func):
    table = {}
    def wrapper(n):
        if n not in table:
            table[n] = func(n)
        return table[n]
    return wrapper

def slow_square(n):
    print("computing", n)
    return n * n

fast = memoised(slow_square)
print(fast(4), fast(4), fast(5))

> **Key idea.** A closure keeps the variables it uses from enclosing functions in cells that outlive the outer call; f.__code__.co_freevars names them and f.__closure__ holds them. The variable is shared, not copied, so later changes are seen. Assigning to an enclosing variable needs nonlocal (otherwise the assignment makes it local); mutating an object it holds doesn't. Functions made in a loop all see the loop variable's final value: fix it with a default argument or, more clearly, a factory function. Use a closure for one operation with a little private state, a class for several operations or inspectable state, and avoid global.

### Exercise: A counter that can reset

Write `make_counter(start=0, step=1)`: returns a function that, each call, returns the next value (`start`, `start + step`, ...), using `nonlocal`. Then `make_resettable(start=0)`: returns a pair `(next_value, reset)` of functions sharing the same count, where `reset()` puts it back to `start`.

In [ ]:
def make_counter(start=0, step=1):
    ___

def make_resettable(start=0):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_counter():\n    \"nonlocal\"\n    c = make_counter()\n    assert [c(), c(), c()] == [0, 1, 2]\n    d = make_counter(10, 5)\n    assert [d(), d()] == [10, 15] and c() == 3\n    nxt, reset = make_resettable(1)\n    assert [nxt(), nxt()] == [1, 2]\n    reset()\n    assert nxt() == 1\n    assert \"nonlocal\" in source and \"[0]\" not in source and \"global\" not in source")

### Exercise: Running statistics

Write `make_tracker()`: returns a function `add(x)` that records a number and returns a tuple `(count, mean, lowest, highest)` of everything added so far (mean rounded to 2 places), keeping only four numbers of state in the closure (not a list of everything).

In [ ]:
def make_tracker():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_tracker():\n    \"state in a closure\"\n    add = make_tracker()\n    assert add(10) == (1, 10.0, 10, 10) and add(20) == (2, 15.0, 10, 20) and add(-5) == (3, 8.33, -5, 20)\n    other = make_tracker()\n    assert other(1) == (1, 1.0, 1, 1)\n    rng = random.Random(1)\n    t, seen = make_tracker(), []\n    for _ in range(100):\n        x = rng.randint(-50, 50)\n        seen.append(x)\n        assert t(x) == (len(seen), round(sum(seen) / len(seen), 2), min(seen), max(seen))\n    assert \".append\" not in source and \"nonlocal\" in source")

### Exercise: Functions made in a loop

Write `multipliers(n)`: a list of n functions, where the k-th (from 0) multiplies its argument by k, built **in a loop** with a factory function (not a default argument). Then `greeters(names)`: a list of functions, each taking no arguments and returning `"Hello, <name>!"` for its own name, built with a list comprehension and a **default argument**.

In [ ]:
def multipliers(n):
    ___

def greeters(names):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_loop():\n    \"late binding avoided\"\n    fs = multipliers(4)\n    assert [f(10) for f in fs] == [0, 10, 20, 30] and multipliers(0) == []\n    gs = greeters([\"Ada\", \"Ben\"])\n    assert [g() for g in gs] == [\"Hello, Ada!\", \"Hello, Ben!\"]\n    assert \"for \" in source.split(\"def multipliers\")[1].split(\"def greeters\")[0]")

### Exercise: Only once

Write `once(func)`: returns a function that calls `func` with the given arguments the **first** time and remembers the result; every later call returns that same result without calling `func` again (whatever the arguments). Then `memoised(func)`: a closure keeping a dict from argument tuples to results, so `func` runs once per distinct set of positional arguments; also give the returned function an attribute `misses`: the number of times `func` actually ran, updated as it goes.

In [ ]:
def once(func):
    ___

def memoised(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_once():\n    \"closures around functions\"\n    runs = []\n    def setup(x):\n        runs.append(x)\n        return x * 2\n    s = once(setup)\n    assert s(5) == 10 and s(7) == 10 and runs == [5]\n    def slow_add(a, b):\n        runs.append((a, b))\n        return a + b\n    fast = memoised(slow_add)\n    assert fast(1, 2) == 3 and fast(1, 2) == 3 and fast(2, 1) == 3 and fast.misses == 2\n    nothing = once(lambda: None)\n    assert nothing() is None and nothing() is None")

### Exercise: Looking inside a closure

Write `remembered(func)`: a dict from each free variable name of `func` to the value in its cell (an empty dict for a function with no closure), using `__code__.co_freevars` and `__closure__`.

In [ ]:
def remembered(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_inspect():\n    \"cells\"\n    def outer(a, b):\n        c = a + b\n        def inner():\n            return a * c\n        return inner\n    assert remembered(outer(2, 3)) == {\"a\": 2, \"c\": 5}\n    assert remembered(len) == {} and remembered(lambda: 1) == {}\n    def counter():\n        n = 0\n        def step():\n            nonlocal n\n            n += 1\n            return n\n        return step\n    s = counter()\n    s()\n    s()\n    assert remembered(s) == {\"n\": 2}")

### Bug hunt: UnboundLocalError

`total += amount` assigns to `total`, which makes it local to `add`, so reading it first fails with `UnboundLocalError`. Declare it `nonlocal` so this prints `[5, 12]`.

In [ ]:
def make_till():
    total = 0
    def add(amount):
        total += amount
        return total
    return add

till = make_till()
try:
    print([till(5), till(7)])
except UnboundLocalError:
    print("UnboundLocalError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"nonlocal\"\n    expect_output(\"[5, 12]\")")

### Bug hunt: Every button says the same

Each callback should report its own button's label, but they all use the loop variable's final value and this prints `['Quit', 'Quit', 'Quit']`. Freeze each label with a default argument so it prints `['Open', 'Save', 'Quit']`.

In [ ]:
callbacks = []
for label in ["Open", "Save", "Quit"]:
    callbacks.append(lambda: label)
print([cb() for cb in callbacks])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one label each\"\n    expect_output(\"['Open', 'Save', 'Quit']\")")

### Bug hunt: State that leaks between counters

The count lives at **module** level with `global`, so two counters share it and this prints `[1, 2, 3]` instead of two independent counts. Keep the count inside `make_counter` with `nonlocal` so it prints `[1, 2, 1]`.

In [ ]:
count = 0

def make_counter():
    def step():
        global count
        count += 1
        return count
    return step

a, b = make_counter(), make_counter()
print([a(), a(), b()])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"private state\"\n    expect_output(\"[1, 2, 1]\")")

### Puzzle: An accumulator

Arrange the lines of a closure that adds each value to a running total and returns the new total. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # [f() for f in [lambda: i for i in range(3)]]
q = ___   # [f() for f in [lambda i=i: i for i in range(3)]]
r = ___   # True or False: appending to an enclosing list needs nonlocal
s = ___   # what make_counter()() returns, for today's warm-up make_counter
t = ___   # the exception raised by `count += 1` inside an inner function without nonlocal

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [2, 2, 2] and q == [0, 1, 2] and r is False, \"p, q and r\"\n    assert s == 1 and t == \"UnboundLocalError\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: undo and redo

Write `make_editor()`: returns a dict of functions sharing one private text and history, with no class and no global:

- `type(s)`: append s to the text.
- `delete(n)`: remove the last n characters (all of them if there are fewer).
- `undo()` and `redo()`: undo the last change or redo the last undone one; each returns True if there was something to undo or redo. Any new `type` or `delete` clears the redo history.
- `text()`: the current text; `history()`: how many changes can be undone.

Keep the state in the enclosing function: the text, a list of past texts for undo, a list for redo. Use `nonlocal` for the text. Then, in a comment at the end, say whether you'd rewrite it as a class, and why.

In [ ]:
def make_editor():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_editor():\n    \"closures sharing state\"\n    ed = make_editor()\n    ed[\"type\"](\"Hello\")\n    ed[\"type\"](\", world\")\n    ed[\"delete\"](6)\n    assert ed[\"text\"]() == \"Hello,\" and ed[\"history\"]() == 3\n    assert ed[\"undo\"]() and ed[\"text\"]() == \"Hello, world\"\n    assert ed[\"undo\"]() and ed[\"undo\"]() and ed[\"text\"]() == \"\" and not ed[\"undo\"]()\n    assert ed[\"redo\"]() and ed[\"text\"]() == \"Hello\"\n    ed[\"type\"](\"!\")\n    assert not ed[\"redo\"]() and ed[\"text\"]() == \"Hello!\"\n    ed[\"delete\"](100)\n    assert ed[\"text\"]() == \"\" and ed[\"undo\"]() and ed[\"text\"]() == \"Hello!\"\n    other = make_editor()\n    assert other[\"text\"]() == \"\" and other[\"history\"]() == 0\n    import re\n    assert not re.search(r\"^\\s*class\\s\", source, re.M) and not re.search(r\"^\\s*global\\s\", source, re.M) and \"nonlocal\" in source")

## Warm-up

Each call of `make_counter` makes a fresh `count`, kept alive for its own `step`. `nonlocal count` lets `step` change it. And Python will show you what a closure remembers.

## What a closure keeps

When an inner function uses a variable of an enclosing function (the **E** in Week 6's LEGB rule), Python doesn't copy the value: it keeps the variable itself in a **cell**, shared by the enclosing function and every inner function that uses it. When the outer function returns, its other local variables disappear, but the cells live on as long as some inner function needs them.

You can look inside: `f.__code__.co_freevars` lists the names a function uses from enclosing scopes ("free variables"), and `f.__closure__` is a tuple of cells, each with a `cell_contents`. Because the **variable** is shared, not the value, two inner functions made in the same call see each other's changes, and an inner function sees a later change made by the outer function.

## nonlocal

Reading an enclosing variable needs nothing special. **Assigning** to it does: an assignment anywhere in a function (including `+=`) makes that name **local** to the function for its whole body. So this fails:

```python
def make_counter():
    count = 0
    def step():
        count += 1        # UnboundLocalError: count is local here, and has no value yet
        return count
    return step
```

`nonlocal count` says "count belongs to the nearest enclosing function; assign to that one". It's the counterpart of `global` for enclosing functions (PEP 3104, Python 3.0, by Ka-Ping Yee). Rules worth knowing:

- `nonlocal` needs an existing variable in some **enclosing function**; it can't create one, and it can't refer to a module-level variable.
- You **don't** need it to **change a mutable object**: `items.append(x)` doesn't assign to `items`. That's why earlier lessons got away with one-item lists (`count = [0]`, `count[0] += 1`): mutating, not assigning. `nonlocal` is the clearer way.

## The loop trap

Because a closure keeps the **variable**, not its value at the time, functions made in a loop that use the loop variable all see its **final** value:

```python
adders = [lambda x: x + i for i in range(3)]
[f(10) for f in adders]       # [12, 12, 12], not [10, 11, 12]
```

Every lambda refers to the same `i`, which is 2 by the time any of them is called. (Wednesday's generator expressions have the same "evaluated later" behaviour for the same reason.) Two fixes:

- **A default argument**, evaluated once, when the function is **defined**: `lambda x, i=i: x + i`. Short, common, but adds a parameter a caller could override.
- **A factory function** that takes the value as a parameter, so each call makes a new variable: `def adder(i): return lambda x: x + i`, then `[adder(i) for i in range(3)]`. Clearer, and it's what `functools.partial` (Thursday) does for you.

## State without a class

A closure is a function with private state. Anything a closure can do, a small class with `__call__` (Week 31) can do too; choosing between them is about clarity:

- **Closure**: one operation, a little state, created in one place: counters, accumulators, memo tables, "call this only once", callbacks that remember a setting. Short, and the state is truly private: nothing outside can reach it except through the function.
- **Class**: several operations on the same state, state that callers should be able to inspect, or anything that will grow. A closure that returns a dict of three inner functions is usually a class in disguise.

A memo table in a closure is the idea behind `functools.cache`, which you've used since Week 16 and which tomorrow shows how to build:

## global, and why not

`global name` lets a function assign to a **module-level** variable. It works, but it makes every function that uses the variable depend on hidden shared state, makes tests interfere with one another, and is a common source of bugs. Prefer, in order: passing values in and returning results; a closure, when a function needs private state of its own; an object, when several functions share state. Constants defined at module level and only **read** never need `global`.

## Practice

## Quiz

## Daily challenge

## Before you go

Find a place in an earlier exercise where a one-item list (`count = [0]`) was used to change a remembered value, and rewrite it with `nonlocal`. Tomorrow: **decorators**, which are closures that wrap functions.